# ⚖️ AI Legal Research Agent
### Powered by LangGraph + Groq LLaMA 3.1 + Serper API

---

**Architecture Overview:**
- 🔍 **Case Search Agent** — Searches legal databases & web for relevant cases
- 📄 **Legal Document Retrieval Agent** — Retrieves and processes legal documents
- ⚖️ **Legal Reasoning Agent** — Analyzes judgments and legal principles
- 📝 **Summary Agent** — Synthesizes insights into actionable summaries

**Workflow:** Legal Question → Search Legal Database → Retrieve Relevant Cases → Analyze Judgments → Summarize Legal Insight

## 📦 Step 1: Install Required Libraries

In [1]:
# Install all required packages
!pip install -q langgraph langchain langchain-groq langchain-community
!pip install -q gradio>=4.0.0
!pip install -q requests beautifulsoup4 httpx
!pip install -q pydantic typing-extensions
!pip install -q faiss-cpu sentence-transformers
!pip install -q tiktoken

print("✅ All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.7 MB/s eta 0:00:00
✅ All packages installed successfully!


## 🔑 Step 2: Import Libraries & Configure API Keys

In [2]:
# ============================================================
# IMPORTS
# ============================================================
import os
import json
import time
import requests
import re
from datetime import datetime
from typing import TypedDict, Annotated, List, Dict, Any, Optional
from dataclasses import dataclass, field

# LangChain & LangGraph
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# Utilities
import gradio as gr
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"📅 Session started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All libraries imported successfully!
📅 Session started: 2026-03-10 18:23:04


In [3]:
# ============================================================
# API KEY CONFIGURATION
# ============================================================

# 🔑 Enter your API Keys here
GROQ_API_KEY    = "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"       # Get from: https://console.groq.com
SERPER_API_KEY  = "34cf9bb4a6965f4ad9bcce86f48f5d693048ac09"      # Get from: https://serper.dev

# Set environment variables
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

# ============================================================
# MODEL CONFIGURATION
# ============================================================
MODEL_NAME     = "llama-3.1-8b-instant"  # Groq LLaMA 3.1 8B
MAX_TOKENS     = 4096
TEMPERATURE    = 0.1

# Initialize the LLM
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    api_key=GROQ_API_KEY
)

print(f"✅ LLM Initialized: {MODEL_NAME}")
print(f"🌡️  Temperature: {TEMPERATURE}")
print(f"📊 Max Tokens: {MAX_TOKENS}")

✅ LLM Initialized: llama-3.1-8b-instant
🌡️  Temperature: 0.1
📊 Max Tokens: 4096


## 🛠️ Step 3: Define Legal Research Tools

In [4]:
# ============================================================
# TOOL 1: SERPER WEB SEARCH
# ============================================================
@tool
def search_legal_web(query: str) -> str:
    """
    Search the web for legal cases, statutes, and legal information using Serper API.
    Best for finding recent cases, legal news, and court decisions.

    Args:
        query: Legal search query (e.g., 'landmark Supreme Court contract law cases')
    Returns:
        Formatted search results with titles, snippets, and links
    """
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json"
    }
    # Add legal context to query
    legal_query = f"{query} legal case law judgment site:law.cornell.edu OR site:justia.com OR site:courtlistener.com OR site:scholar.google.com"
    payload = {"q": legal_query, "num": 8, "gl": "us", "hl": "en"}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        data = response.json()

        results = []
        if "organic" in data:
            for i, result in enumerate(data["organic"][:6], 1):
                results.append(
                    f"[{i}] **{result.get('title', 'N/A')}**\n"
                    f"    🔗 {result.get('link', 'N/A')}\n"
                    f"    📋 {result.get('snippet', 'No description available')}\n"
                )

        # Also capture knowledge graph if available
        if "knowledgeGraph" in data:
            kg = data["knowledgeGraph"]
            results.insert(0, f"📚 **Knowledge Panel:** {kg.get('title', '')} — {kg.get('description', '')}\n")

        return "\n".join(results) if results else "No results found for this legal query."

    except Exception as e:
        return f"Search error: {str(e)}. Please verify your Serper API key."


# ============================================================
# TOOL 2: LEGAL DATABASE SEARCH (CourtListener / Free Law)
# ============================================================
@tool
def search_case_law_database(query: str, jurisdiction: str = "all") -> str:
    """
    Search CourtListener's free legal database for case law and opinions.

    Args:
        query: Legal search terms or case name
        jurisdiction: Jurisdiction filter (e.g., 'scotus', 'ca9', 'all')
    Returns:
        List of matching legal cases with citations
    """
    base_url = "https://www.courtlistener.com/api/rest/v3/search/"
    params = {
        "q": query,
        "type": "o",  # opinions
        "order_by": "score desc",
        "format": "json"
    }
    if jurisdiction != "all":
        params["court"] = jurisdiction

    try:
        response = requests.get(base_url, params=params, timeout=15)
        if response.status_code == 200:
            data = response.json()
            results = data.get("results", [])[:5]

            if not results:
                return f"No cases found in CourtListener for: '{query}'"

            formatted = [f"📚 **CourtListener Results for '{query}':**\n"]
            for i, case in enumerate(results, 1):
                case_name   = case.get("caseName", "Unknown Case")
                court       = case.get("court", "Unknown Court")
                date_filed  = case.get("dateFiled", "Unknown Date")
                citation    = case.get("citation", ["No citation"])
                snippet     = case.get("snippet", "No summary available")
                url         = f"https://www.courtlistener.com{case.get('absolute_url', '')}"

                formatted.append(
                    f"[{i}] **{case_name}**\n"
                    f"    🏛️  Court: {court} | 📅 Date: {date_filed}\n"
                    f"    📎 Citation: {', '.join(citation) if isinstance(citation, list) else citation}\n"
                    f"    📝 {BeautifulSoup(snippet, 'html.parser').get_text()[:200]}...\n"
                    f"    🔗 {url}\n"
                )
            return "\n".join(formatted)
        else:
            return f"Database returned status {response.status_code}. Falling back to web search."
    except Exception as e:
        return f"Database search error: {str(e)}"


# ============================================================
# TOOL 3: STATUTE & REGULATION LOOKUP
# ============================================================
@tool
def lookup_statute(statute_query: str) -> str:
    """
    Search for federal statutes, US Code sections, and regulations.
    Uses the Free Law Project API and Cornell LII.

    Args:
        statute_query: Statute name or code reference (e.g., '42 USC 1983', 'ADA Title II')
    Returns:
        Statute text, description, and related regulations
    """
    # Search via Serper for statute info
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json"
    }
    query = f"{statute_query} US statute text site:law.cornell.edu OR site:uscode.house.gov"
    payload = {"q": query, "num": 5}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        data = response.json()

        results = [f"📜 **Statute Search: '{statute_query}'**\n"]
        if "organic" in data:
            for i, result in enumerate(data["organic"][:4], 1):
                results.append(
                    f"[{i}] {result.get('title', 'N/A')}\n"
                    f"    🔗 {result.get('link', 'N/A')}\n"
                    f"    📋 {result.get('snippet', '')}\n"
                )

        # Add answerBox if present
        if "answerBox" in data:
            ab = data["answerBox"]
            results.insert(1, f"\n⚡ **Direct Answer:** {ab.get('answer', ab.get('snippet', ''))}\n")

        return "\n".join(results)
    except Exception as e:
        return f"Statute lookup error: {str(e)}"


# ============================================================
# TOOL 4: LEGAL NEWS & RECENT DEVELOPMENTS
# ============================================================
@tool
def get_legal_news(topic: str, time_period: str = "recent") -> str:
    """
    Fetch recent legal news, court decisions, and regulatory updates.

    Args:
        topic: Legal topic or area of law
        time_period: 'recent', 'this_year', or 'last_month'
    Returns:
        Recent legal developments and news
    """
    url = "https://google.serper.dev/news"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json"
    }
    query = f"{topic} legal ruling court decision 2024 2025"
    payload = {"q": query, "num": 6, "gl": "us"}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        data = response.json()

        results = [f"📰 **Recent Legal Developments: '{topic}'**\n"]
        news_items = data.get("news", [])

        if not news_items:
            return f"No recent news found for '{topic}'"

        for i, item in enumerate(news_items[:5], 1):
            results.append(
                f"[{i}] **{item.get('title', 'N/A')}**\n"
                f"    📰 Source: {item.get('source', 'N/A')} | 📅 {item.get('date', 'N/A')}\n"
                f"    📋 {item.get('snippet', 'No description')}\n"
                f"    🔗 {item.get('link', '')}\n"
            )

        return "\n".join(results)
    except Exception as e:
        return f"News fetch error: {str(e)}"


# ============================================================
# TOOL 5: LEGAL CITATION ANALYZER
# ============================================================
@tool
def analyze_legal_citation(citation: str) -> str:
    """
    Analyze and look up a specific legal citation to get case details.
    Supports Bluebook and common citation formats.

    Args:
        citation: Legal citation in standard format (e.g., '347 U.S. 483', 'Brown v. Board of Education')
    Returns:
        Case details, holding, and significance
    """
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json"
    }
    query = f"{citation} case law holding significance legal analysis"
    payload = {"q": query, "num": 5}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        data = response.json()

        results = [f"⚖️ **Citation Analysis: '{citation}'**\n"]

        if "answerBox" in data:
            ab = data["answerBox"]
            results.append(f"📌 **Quick Answer:** {ab.get('answer', ab.get('snippet', ''))}\n")

        if "knowledgeGraph" in data:
            kg = data["knowledgeGraph"]
            results.append(f"🏛️ **Case:** {kg.get('title', '')}\n📝 {kg.get('description', '')}\n")

        for i, result in enumerate(data.get("organic", [])[:4], 1):
            results.append(
                f"[{i}] {result.get('title', 'N/A')}\n"
                f"    📋 {result.get('snippet', '')}\n"
                f"    🔗 {result.get('link', '')}\n"
            )

        return "\n".join(results)
    except Exception as e:
        return f"Citation analysis error: {str(e)}"


# Collect all tools
LEGAL_TOOLS = [
    search_legal_web,
    search_case_law_database,
    lookup_statute,
    get_legal_news,
    analyze_legal_citation
]

print("✅ Legal Research Tools Initialized:")
for t in LEGAL_TOOLS:
    print(f"   🔧 {t.name}")

✅ Legal Research Tools Initialized:
   🔧 search_legal_web
   🔧 search_case_law_database
   🔧 lookup_statute
   🔧 get_legal_news
   🔧 analyze_legal_citation


## 🤖 Step 4: Define Agent State & LangGraph Nodes

In [5]:
# ============================================================
# AGENT STATE DEFINITION
# ============================================================
class LegalResearchState(TypedDict):
    """State shared across all agents in the Legal Research workflow."""
    # Core inputs
    legal_question:      str
    jurisdiction:        str
    practice_area:       str
    research_depth:      str

    # Agent outputs
    search_results:      str
    retrieved_documents: str
    legal_analysis:      str
    final_summary:       str
    citations:           List[str]

    # Metadata
    messages:            Annotated[List, add_messages]
    current_agent:       str
    agent_notes:         Dict[str, str]
    status:              str
    error:               Optional[str]
    timestamp:           str


print("✅ LegalResearchState defined with fields:")
for field_name, field_type in LegalResearchState.__annotations__.items():
    print(f"   📌 {field_name}: {field_type}")

✅ LegalResearchState defined with fields:
   📌 legal_question: <class 'str'>
   📌 jurisdiction: <class 'str'>
   📌 practice_area: <class 'str'>
   📌 research_depth: <class 'str'>
   📌 search_results: <class 'str'>
   📌 retrieved_documents: <class 'str'>
   📌 legal_analysis: <class 'str'>
   📌 final_summary: <class 'str'>
   📌 citations: typing.List[str]
   📌 messages: typing.Annotated[typing.List, <function _add_messages_wrapper.<locals>._add_messages at 0x7ca94a51b6a0>]
   📌 current_agent: <class 'str'>
   📌 agent_notes: typing.Dict[str, str]
   📌 status: <class 'str'>
   📌 error: typing.Optional[str]
   📌 timestamp: <class 'str'>


In [6]:
# ============================================================
# AGENT 1: CASE SEARCH AGENT
# ============================================================
def case_search_agent(state: LegalResearchState) -> LegalResearchState:
    """
    Agent 1: Searches legal databases and web for relevant cases.
    Uses Serper API and CourtListener to find pertinent case law.
    """
    print("\n🔍 [Agent 1] Case Search Agent — Starting...")

    question    = state["legal_question"]
    jurisdiction = state.get("jurisdiction", "Federal")
    practice_area = state.get("practice_area", "General")
    depth       = state.get("research_depth", "Standard")

    # Determine number of searches based on depth
    search_count = {"Quick": 1, "Standard": 2, "Deep": 3, "Comprehensive": 4}.get(depth, 2)

    system_prompt = f"""You are an expert Legal Research Specialist with 20+ years of experience.
You are conducting a {depth} legal research for a {jurisdiction} jurisdiction matter in the area of {practice_area}.

Your tasks:
1. Identify the key legal issues in the question
2. Determine relevant search terms for case law databases
3. Search for relevant cases, statutes, and precedents
4. Catalog and organize your findings

Use the available tools to search for: landmark cases, recent rulings, applicable statutes.
Perform {search_count} targeted searches to ensure comprehensive coverage."""

    # Bind tools to LLM
    llm_with_tools = llm.bind_tools(LEGAL_TOOLS)

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
Legal Question: {question}
Jurisdiction: {jurisdiction}
Practice Area: {practice_area}
Research Depth: {depth}

Please search for relevant cases and legal authorities. Use multiple tools to gather comprehensive results.
Search for: (1) landmark cases on this topic, (2) recent relevant rulings, (3) applicable statutes.
""")
    ]

    all_results = []

    try:
        # Initial LLM call to get tool calls
        response = llm_with_tools.invoke(messages)

        # Process tool calls if any
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tc in response.tool_calls:
                tool_name = tc["name"]
                tool_args = tc["args"]

                # Execute the tool
                tool_map = {t.name: t for t in LEGAL_TOOLS}
                if tool_name in tool_map:
                    result = tool_map[tool_name].invoke(tool_args)
                    all_results.append(f"\n### Tool: {tool_name}\n{result}")
                    print(f"   ✅ Executed tool: {tool_name}")

        # If no tool calls, do direct searches
        if not all_results:
            web_result = search_legal_web.invoke({"query": f"{question} {jurisdiction} case law"})
            db_result  = search_case_law_database.invoke({"query": question, "jurisdiction": "all"})
            all_results = [f"\n### Web Search\n{web_result}", f"\n### Case Law Database\n{db_result}"]

        search_output = "\n".join(all_results)

        # Summarize findings
        summary_response = llm.invoke([
            SystemMessage(content="You are a legal research expert. Summarize the search findings concisely."),
            HumanMessage(content=f"Question: {question}\n\nSearch Results:\n{search_output}\n\nProvide a structured summary of the most relevant cases found.")
        ])

        final_search_results = f"""## 🔍 Case Search Results

**Legal Question:** {question}
**Jurisdiction:** {jurisdiction} | **Practice Area:** {practice_area}

---

### Raw Search Data:
{search_output}

---

### Agent Summary:
{summary_response.content}
"""

        print("   ✅ Case Search Agent completed successfully")
        return {
            **state,
            "search_results": final_search_results,
            "current_agent": "case_search",
            "status": "search_complete",
            "messages": state["messages"] + [AIMessage(content=f"Case Search completed. Found relevant cases for: {question}")]
        }

    except Exception as e:
        error_msg = f"Case Search Agent error: {str(e)}"
        print(f"   ❌ {error_msg}")
        return {
            **state,
            "search_results": f"Search encountered an error: {str(e)}",
            "current_agent": "case_search",
            "status": "search_error",
            "error": error_msg
        }


# ============================================================
# AGENT 2: LEGAL DOCUMENT RETRIEVAL AGENT
# ============================================================
def document_retrieval_agent(state: LegalResearchState) -> LegalResearchState:
    """
    Agent 2: Retrieves and processes legal documents from search results.
    Extracts key holdings, precedents, and applicable rules.
    """
    print("\n📄 [Agent 2] Document Retrieval Agent — Starting...")

    question     = state["legal_question"]
    search_results = state.get("search_results", "")
    jurisdiction = state.get("jurisdiction", "Federal")
    practice_area = state.get("practice_area", "General")

    system_prompt = f"""You are an expert Legal Document Analyst specializing in {practice_area} law.
Your role is to:
1. Extract key legal holdings from the search results
2. Identify the most relevant cases and their precedential value
3. Retrieve specific statutes and regulations that apply
4. Organize documents by relevance and authority
5. Note any circuit splits or conflicting authorities

Focus on {jurisdiction} jurisdiction. Distinguish between binding and persuasive authority."""

    llm_with_tools = llm.bind_tools([lookup_statute, analyze_legal_citation, get_legal_news])

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
Legal Question: {question}

Previous Search Results:
{search_results[:3000]}

Tasks:
1. Look up the most important statutes mentioned
2. Analyze the key case citations found
3. Get recent legal developments on this topic
4. Compile a structured document repository
""")
    ]

    retrieved_docs = []

    try:
        response = llm_with_tools.invoke(messages)

        # Execute tool calls
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tc in response.tool_calls:
                tool_name = tc["name"]
                tool_args = tc["args"]
                tool_map  = {t.name: t for t in LEGAL_TOOLS}
                if tool_name in tool_map:
                    result = tool_map[tool_name].invoke(tool_args)
                    retrieved_docs.append(f"\n**{tool_name}:**\n{result}")
                    print(f"   ✅ Retrieved: {tool_name}")

        # Always get recent news
        news = get_legal_news.invoke({"topic": f"{question} {practice_area}", "time_period": "recent"})
        retrieved_docs.append(f"\n**Recent Developments:**\n{news}")

        # Synthesize document analysis
        synthesis = llm.invoke([
            SystemMessage(content="You are a legal document analyst. Extract and organize key legal documents."),
            HumanMessage(content=f"""
Question: {question}
Jurisdiction: {jurisdiction}

Retrieved Documents:
{chr(10).join(retrieved_docs)[:4000]}

Please:
1. List the TOP 5 most relevant cases with their holdings
2. Identify key statutes and their relevance
3. Note any binding vs persuasive authority
4. Highlight any recent developments that affect the analysis
Format as a structured legal document repository.
""")
        ])

        final_docs = f"""## 📄 Legal Document Repository

**Practice Area:** {practice_area} | **Jurisdiction:** {jurisdiction}

---

### Retrieved Legal Authorities:
{chr(10).join(retrieved_docs)}

---

### Document Analysis:
{synthesis.content}
"""

        print("   ✅ Document Retrieval Agent completed successfully")
        return {
            **state,
            "retrieved_documents": final_docs,
            "current_agent": "document_retrieval",
            "status": "retrieval_complete",
            "messages": state["messages"] + [AIMessage(content="Document retrieval and analysis complete.")]
        }

    except Exception as e:
        error_msg = f"Document Retrieval error: {str(e)}"
        print(f"   ❌ {error_msg}")
        return {
            **state,
            "retrieved_documents": f"Document retrieval error: {str(e)}",
            "current_agent": "document_retrieval",
            "status": "retrieval_error",
            "error": error_msg
        }


# ============================================================
# AGENT 3: LEGAL REASONING AGENT
# ============================================================
def legal_reasoning_agent(state: LegalResearchState) -> LegalResearchState:
    """
    Agent 3: Applies legal reasoning to analyze cases and derive legal principles.
    Performs IRAC analysis and identifies applicable legal standards.
    """
    print("\n⚖️ [Agent 3] Legal Reasoning Agent — Starting...")

    question     = state["legal_question"]
    search_res   = state.get("search_results", "")
    docs         = state.get("retrieved_documents", "")
    jurisdiction = state.get("jurisdiction", "Federal")
    practice_area = state.get("practice_area", "General")
    depth        = state.get("research_depth", "Standard")

    system_prompt = f"""You are a Senior Legal Counsel and Law Professor with expertise in {practice_area}.
You are conducting a {depth} legal analysis for {jurisdiction} jurisdiction.

Apply rigorous legal reasoning using:
- IRAC Framework (Issue, Rule, Application, Conclusion)
- Analogical reasoning from precedents
- Statutory interpretation principles
- Policy considerations
- Identification of legal tests and standards

Be precise, cite authorities properly, and acknowledge uncertainty where it exists.
Identify both strong and weak points in the legal argument."""

    analysis_prompt = f"""
LEGAL QUESTION: {question}

JURISDICTION: {jurisdiction}
PRACTICE AREA: {practice_area}

RESEARCH FINDINGS:
{search_res[:2000]}

RETRIEVED DOCUMENTS:
{docs[:2000]}

Please provide a comprehensive legal analysis:

## 1. ISSUE IDENTIFICATION
Identify the precise legal issues raised by this question.

## 2. APPLICABLE LEGAL RULES
State the controlling rules, statutes, and common law principles.

## 3. CASE LAW ANALYSIS
Analyze the most relevant precedents:
- Landmark cases and their holdings
- How courts have applied these rules
- Any circuit splits or conflicting authorities

## 4. APPLICATION & REASONING
Apply the rules to the legal question with rigorous reasoning.

## 5. LEGAL STANDARDS & TESTS
Identify specific legal tests (e.g., balancing tests, strict scrutiny, reasonableness standards).

## 6. COUNTERARGUMENTS & RISKS
Present opposing arguments and legal risks.

## 7. PRELIMINARY CONCLUSION
State your legal conclusion with confidence level (High/Medium/Low).
"""

    try:
        analysis_response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=analysis_prompt)
        ])

        # Extract citations from the analysis
        analysis_text = analysis_response.content

        # Simple citation extraction (can be enhanced with regex)
        citations = re.findall(r'\d+\s+[A-Z][a-zA-Z\.]+\s+\d+', analysis_text)
        citations += re.findall(r'[A-Z][a-zA-Z]+\s+v\.\s+[A-Z][a-zA-Z]+', analysis_text)
        citations = list(set(citations))[:10]  # Deduplicate

        final_analysis = f"""## ⚖️ Legal Reasoning Analysis

**Analyzed by:** Legal Reasoning Agent (LLaMA 3.1 8B)
**Question:** {question}
**Jurisdiction:** {jurisdiction} | **Depth:** {depth}

---

{analysis_text}

---

**Citations Identified:** {', '.join(citations) if citations else 'See analysis above'}
"""

        print("   ✅ Legal Reasoning Agent completed successfully")
        return {
            **state,
            "legal_analysis": final_analysis,
            "citations": citations,
            "current_agent": "legal_reasoning",
            "status": "analysis_complete",
            "messages": state["messages"] + [AIMessage(content="Legal reasoning and IRAC analysis complete.")]
        }

    except Exception as e:
        error_msg = f"Legal Reasoning error: {str(e)}"
        print(f"   ❌ {error_msg}")
        return {
            **state,
            "legal_analysis": f"Reasoning error: {str(e)}",
            "current_agent": "legal_reasoning",
            "status": "reasoning_error",
            "error": error_msg
        }


# ============================================================
# AGENT 4: SUMMARY AGENT
# ============================================================
def summary_agent(state: LegalResearchState) -> LegalResearchState:
    """
    Agent 4: Synthesizes all research into a professional legal memo/summary.
    Produces actionable insights for law firms and compliance teams.
    """
    print("\n📝 [Agent 4] Summary Agent — Starting...")

    question     = state["legal_question"]
    search_res   = state.get("search_results", "")
    docs         = state.get("retrieved_documents", "")
    analysis     = state.get("legal_analysis", "")
    citations    = state.get("citations", [])
    jurisdiction = state.get("jurisdiction", "Federal")
    practice_area = state.get("practice_area", "General")
    depth        = state.get("research_depth", "Standard")
    timestamp    = state.get("timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

    system_prompt = f"""You are a Senior Partner at a prestigious law firm writing a professional legal research memorandum.
Your memo must be:
- Clear and accessible to both lawyers and clients
- Organized with proper legal memo structure
- Actionable with specific recommendations
- Properly citing authorities
- Noting limitations and areas of uncertainty

Format as a formal Legal Research Memo for {jurisdiction} jurisdiction in {practice_area} law."""

    summary_prompt = f"""
Create a comprehensive Legal Research Memorandum based on:

QUESTION: {question}

RESEARCH CONDUCTED:
{search_res[:1500]}

DOCUMENTS RETRIEVED:
{docs[:1500]}

LEGAL ANALYSIS:
{analysis[:2000]}

Structure the memo as follows:

═══════════════════════════════════════
LEGAL RESEARCH MEMORANDUM
═══════════════════════════════════════
TO: [Client / Requesting Party]
FROM: AI Legal Research System
DATE: {timestamp}
RE: {question}
JURISDICTION: {jurisdiction}
PRACTICE AREA: {practice_area}
RESEARCH DEPTH: {depth}
═══════════════════════════════════════

I. EXECUTIVE SUMMARY (2-3 sentences)

II. QUESTION PRESENTED

III. BRIEF ANSWER

IV. KEY LEGAL AUTHORITIES
   A. Controlling Cases
   B. Relevant Statutes
   C. Secondary Authorities

V. DISCUSSION
   A. Legal Framework
   B. Analysis
   C. Application

VI. RISKS & CONSIDERATIONS

VII. RECOMMENDATIONS

VIII. CONCLUSION

IX. DISCLAIMER
"""

    try:
        summary_response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=summary_prompt)
        ])

        final_memo = summary_response.content

        # Add research metadata footer
        research_metadata = f"""

---
**Research Metadata:**
- Model: {MODEL_NAME} (via Groq)
- Agents Used: Case Search → Document Retrieval → Legal Reasoning → Summary
- Citations Found: {len(citations)}
- Generated: {timestamp}
- ⚠️ This memo is AI-generated. Always consult a licensed attorney for legal advice.
"""

        final_output = final_memo + research_metadata

        print("   ✅ Summary Agent completed successfully")
        print("\n🎉 All agents completed. Legal Research Memo ready!")

        return {
            **state,
            "final_summary": final_output,
            "current_agent": "summary",
            "status": "complete",
            "messages": state["messages"] + [AIMessage(content="Legal Research Memorandum completed.")]
        }

    except Exception as e:
        error_msg = f"Summary Agent error: {str(e)}"
        print(f"   ❌ {error_msg}")
        return {
            **state,
            "final_summary": f"Summary generation error: {str(e)}",
            "current_agent": "summary",
            "status": "summary_error",
            "error": error_msg
        }


print("✅ All 4 Agent functions defined:")
print("   🔍 Agent 1: case_search_agent")
print("   📄 Agent 2: document_retrieval_agent")
print("   ⚖️  Agent 3: legal_reasoning_agent")
print("   📝 Agent 4: summary_agent")

✅ All 4 Agent functions defined:
   🔍 Agent 1: case_search_agent
   📄 Agent 2: document_retrieval_agent
   ⚖️  Agent 3: legal_reasoning_agent
   📝 Agent 4: summary_agent


## 🕸️ Step 5: Build the LangGraph Workflow

In [7]:
# ============================================================
# BUILD THE LANGGRAPH WORKFLOW
# ============================================================

def build_legal_research_graph():
    """Build and compile the multi-agent LangGraph workflow."""

    # Create the state graph
    workflow = StateGraph(LegalResearchState)

    # Add all agent nodes
    workflow.add_node("case_search",         case_search_agent)
    workflow.add_node("document_retrieval",  document_retrieval_agent)
    workflow.add_node("legal_reasoning",     legal_reasoning_agent)
    workflow.add_node("summary",             summary_agent)

    # Define the workflow edges (sequential pipeline)
    workflow.add_edge(START,               "case_search")
    workflow.add_edge("case_search",       "document_retrieval")
    workflow.add_edge("document_retrieval","legal_reasoning")
    workflow.add_edge("legal_reasoning",   "summary")
    workflow.add_edge("summary",           END)

    # Compile the graph
    graph = workflow.compile()

    return graph


# Compile the graph
legal_research_graph = build_legal_research_graph()

print("✅ LangGraph Workflow compiled successfully!")
print("\n📊 Workflow Architecture:")
print("   START")
print("     ↓")
print("   🔍 case_search (Agent 1)")
print("     ↓")
print("   📄 document_retrieval (Agent 2)")
print("     ↓")
print("   ⚖️  legal_reasoning (Agent 3)")
print("     ↓")
print("   📝 summary (Agent 4)")
print("     ↓")
print("   END")

✅ LangGraph Workflow compiled successfully!

📊 Workflow Architecture:
   START
     ↓
   🔍 case_search (Agent 1)
     ↓
   📄 document_retrieval (Agent 2)
     ↓
   ⚖️  legal_reasoning (Agent 3)
     ↓
   📝 summary (Agent 4)
     ↓
   END


In [8]:
# ============================================================
# MAIN EXECUTION FUNCTION
# ============================================================

def run_legal_research(
    legal_question: str,
    jurisdiction: str = "Federal",
    practice_area: str = "General Civil",
    research_depth: str = "Standard"
) -> Dict[str, Any]:
    """
    Execute the full legal research pipeline.

    Args:
        legal_question: The legal question to research
        jurisdiction: Legal jurisdiction (Federal, California, New York, etc.)
        practice_area: Area of law (Contract Law, Constitutional Law, etc.)
        research_depth: Depth of research (Quick, Standard, Deep, Comprehensive)

    Returns:
        Dict with all agent outputs and final memo
    """
    if not legal_question.strip():
        return {"error": "Please provide a legal question"}

    # Initialize state
    initial_state: LegalResearchState = {
        "legal_question":      legal_question,
        "jurisdiction":        jurisdiction,
        "practice_area":       practice_area,
        "research_depth":      research_depth,
        "search_results":      "",
        "retrieved_documents": "",
        "legal_analysis":      "",
        "final_summary":       "",
        "citations":           [],
        "messages":            [HumanMessage(content=legal_question)],
        "current_agent":       "initializing",
        "agent_notes":         {},
        "status":              "started",
        "error":               None,
        "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    start_time = time.time()
    print(f"\n{'='*60}")
    print(f"⚖️  LEGAL RESEARCH INITIATED")
    print(f"{'='*60}")
    print(f"❓ Question: {legal_question[:100]}...")
    print(f"🏛️  Jurisdiction: {jurisdiction}")
    print(f"📚 Practice Area: {practice_area}")
    print(f"🔬 Research Depth: {research_depth}")
    print(f"{'='*60}")

    # Execute the graph
    final_state = legal_research_graph.invoke(initial_state)

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ RESEARCH COMPLETE in {elapsed:.1f} seconds")
    print(f"{'='*60}")

    return {
        "final_memo":       final_state.get("final_summary", ""),
        "search_results":   final_state.get("search_results", ""),
        "documents":        final_state.get("retrieved_documents", ""),
        "analysis":         final_state.get("legal_analysis", ""),
        "citations":        final_state.get("citations", []),
        "status":           final_state.get("status", "unknown"),
        "elapsed_seconds":  elapsed
    }


print("✅ run_legal_research() function ready!")

✅ run_legal_research() function ready!


## 🎨 Step 6: Build the Professional Gradio UI

In [9]:
# ============================================================
# PROFESSIONAL GRADIO UI
# ============================================================

# Custom CSS for professional legal theme
CUSTOM_CSS = """
/* ===== ROOT VARIABLES ===== */
:root {
    --primary:     #1a3a5c;
    --accent:      #c9a84c;
    --light-bg:    #f7f5f0;
    --card-bg:     #ffffff;
    --border:      #d4c9b0;
    --text:        #2c2c2c;
    --text-light:  #666;
    --success:     #2e7d32;
    --info:        #1565c0;
    --shadow:      0 2px 12px rgba(26,58,92,0.10);
}

/* ===== BODY & CONTAINER ===== */
body, .gradio-container {
    background: var(--light-bg) !important;
    font-family: 'Georgia', 'Times New Roman', serif !important;
    color: var(--text) !important;
}

/* ===== HEADER ===== */
.app-header {
    background: linear-gradient(135deg, #1a3a5c 0%, #0d2035 60%, #1a3a5c 100%);
    padding: 30px 40px;
    border-radius: 12px;
    margin-bottom: 24px;
    text-align: center;
    box-shadow: 0 4px 20px rgba(26,58,92,0.25);
    border-bottom: 4px solid var(--accent);
    position: relative;
    overflow: hidden;
}

.app-header::before {
    content: '';
    position: absolute;
    top: 0; left: 0; right: 0; bottom: 0;
    background: repeating-linear-gradient(
        45deg,
        transparent,
        transparent 30px,
        rgba(201,168,76,0.03) 30px,
        rgba(201,168,76,0.03) 60px
    );
}

.app-header h1 {
    color: #ffffff !important;
    font-size: 2.2em !important;
    font-weight: bold !important;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-bottom: 8px !important;
}

.app-header p {
    color: var(--accent) !important;
    font-size: 1.05em !important;
    letter-spacing: 1px;
}

/* ===== SECTION CARDS ===== */
.section-card {
    background: var(--card-bg);
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 20px 24px;
    margin-bottom: 18px;
    box-shadow: var(--shadow);
}

.section-title {
    color: var(--primary) !important;
    font-size: 1.1em !important;
    font-weight: bold !important;
    border-bottom: 2px solid var(--accent);
    padding-bottom: 8px;
    margin-bottom: 16px !important;
    text-transform: uppercase;
    letter-spacing: 1px;
}

/* ===== BUTTONS ===== */
#research-btn {
    background: linear-gradient(135deg, #1a3a5c, #0d2035) !important;
    color: white !important;
    font-size: 1.15em !important;
    font-weight: bold !important;
    border: 2px solid var(--accent) !important;
    border-radius: 8px !important;
    padding: 14px 28px !important;
    letter-spacing: 1px;
    text-transform: uppercase;
    cursor: pointer;
    transition: all 0.2s ease;
    box-shadow: 0 4px 15px rgba(26,58,92,0.3) !important;
}
#research-btn:hover {
    background: linear-gradient(135deg, #c9a84c, #a88030) !important;
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(201,168,76,0.4) !important;
}

#clear-btn {
    background: #f0ece4 !important;
    color: var(--primary) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    font-weight: bold !important;
}

#example-btn {
    background: linear-gradient(135deg, #e8f4f8, #d0e8f4) !important;
    color: var(--info) !important;
    border: 1px solid #90caf9 !important;
    border-radius: 8px !important;
    font-weight: bold !important;
}

/* ===== INPUT FIELDS ===== */
textarea, input[type=text], .gr-input {
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    font-family: 'Georgia', serif !important;
    background: #fdfcf9 !important;
}

textarea:focus, input:focus {
    border-color: var(--accent) !important;
    box-shadow: 0 0 0 2px rgba(201,168,76,0.2) !important;
    outline: none !important;
}

/* ===== DROPDOWN ===== */
.gr-dropdown, select {
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    background: #fdfcf9 !important;
}

/* ===== TABS ===== */
.tab-nav {
    background: #eee9de !important;
    border-bottom: 2px solid var(--accent) !important;
    border-radius: 8px 8px 0 0 !important;
}

.tab-nav button {
    color: var(--primary) !important;
    font-weight: bold !important;
    font-family: 'Georgia', serif !important;
    padding: 10px 20px !important;
    border-radius: 6px 6px 0 0 !important;
}

.tab-nav button.selected {
    background: var(--primary) !important;
    color: white !important;
    border-bottom: 3px solid var(--accent) !important;
}

/* ===== PROGRESS / STATUS ===== */
.status-bar {
    background: linear-gradient(90deg, #eaf3eb, #f7fdf7);
    border: 1px solid #81c784;
    border-left: 4px solid #2e7d32;
    border-radius: 6px;
    padding: 10px 16px;
    font-size: 0.95em;
    color: #1b5e20;
}

/* ===== OUTPUT AREAS ===== */
.gr-markdown, .output-text {
    background: #fdfcf9 !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    padding: 16px !important;
    line-height: 1.7 !important;
    font-family: 'Georgia', serif !important;
}

/* ===== AGENT BADGES ===== */
.agent-badge {
    display: inline-block;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: 0.85em;
    font-weight: bold;
    margin-right: 8px;
    margin-bottom: 4px;
}

/* ===== FOOTER ===== */
.app-footer {
    text-align: center;
    color: var(--text-light);
    font-size: 0.85em;
    padding: 16px;
    border-top: 1px solid var(--border);
    margin-top: 24px;
}

/* ===== METRICS ROW ===== */
.metrics-row {
    display: flex;
    gap: 12px;
    flex-wrap: wrap;
}

.metric-card {
    background: linear-gradient(135deg, #f7f5f0, #ede9de);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 12px 16px;
    flex: 1;
    min-width: 100px;
    text-align: center;
    box-shadow: 0 1px 4px rgba(0,0,0,0.06);
}
"""


# ============================================================
# GRADIO PROCESSING FUNCTION
# ============================================================
def process_legal_research(
    legal_question, jurisdiction, practice_area, research_depth,
    include_news, include_statutes, output_format
):
    """Main processing function called by Gradio UI."""

    if not legal_question.strip():
        error_msg = "⚠️ Please enter a legal question to research."
        return error_msg, error_msg, error_msg, error_msg, "❌ No question provided"

    # Enhance the question based on options
    enhanced_question = legal_question
    if include_news:
        enhanced_question += " Include recent legal developments and news."
    if include_statutes:
        enhanced_question += " Include relevant statutes and regulations."

    status_msg = f"🔄 Running {research_depth} research on: {legal_question[:60]}..."

    try:
        results = run_legal_research(
            legal_question  = enhanced_question,
            jurisdiction    = jurisdiction,
            practice_area   = practice_area,
            research_depth  = research_depth
        )

        if "error" in results:
            err = results["error"]
            return err, err, err, err, f"❌ Error: {err}"

        elapsed = results.get("elapsed_seconds", 0)
        citations_count = len(results.get("citations", []))

        status = (
            f"✅ Research Complete | "
            f"⏱️ {elapsed:.1f}s | "
            f"📎 {citations_count} citations | "
            f"🔬 {research_depth} depth | "
            f"🏛️ {jurisdiction}"
        )

        # Format output based on selected format
        final_memo = results.get("final_memo", "No memo generated")
        if output_format == "Plain Text":
            # Strip markdown for plain text
            final_memo = re.sub(r'[#*`_]', '', final_memo)

        return (
            results.get("final_memo", "No memo generated"),
            results.get("search_results", "No search results"),
            results.get("documents", "No documents retrieved"),
            results.get("analysis", "No analysis performed"),
            status
        )

    except Exception as e:
        err = f"❌ System Error: {str(e)}\n\nPlease check your API keys and try again."
        return err, err, err, err, f"❌ Error: {str(e)[:80]}"


def load_example(example_name):
    """Load a pre-defined legal research example."""
    examples = {
        "Contract Breach (Federal)": (
            "What are the legal requirements for proving breach of contract, and what damages can be recovered when a contractor fails to deliver software on time?",
            "Federal", "Contract Law", "Standard"
        ),
        "Employment Discrimination (CA)": (
            "Can an employer be held liable for racial discrimination if a manager makes racially biased promotion decisions without explicit company policy?",
            "California", "Employment Law", "Deep"
        ),
        "Fourth Amendment Search (SCOTUS)": (
            "What Fourth Amendment protections apply to digital data stored in cloud services, and can police access this data without a warrant?",
            "Federal", "Constitutional Law", "Comprehensive"
        ),
        "Corporate Liability (Delaware)": (
            "Under what circumstances can corporate directors be personally liable for breach of fiduciary duty in a merger transaction?",
            "Delaware", "Corporate Law", "Standard"
        ),
        "IP Copyright Infringement": (
            "What constitutes fair use in copyright law, and how do courts determine whether AI-generated content using copyrighted training data infringes copyright?",
            "Federal", "Intellectual Property", "Deep"
        )
    }
    if example_name in examples:
        q, j, p, d = examples[example_name]
        return q, j, p, d
    return "", "Federal", "General Civil", "Standard"


print("✅ Gradio UI functions defined!")

✅ Gradio UI functions defined!


In [10]:
# ============================================================
# BUILD & LAUNCH THE GRADIO INTERFACE
# ============================================================

HEADER_HTML = """
<div class="app-header">
    <h1>⚖️ AI Legal Research Agent</h1>
    <p>Powered by LangGraph • LLaMA 3.1 8B (Groq) • Serper Legal Search</p>
    <p style="color: rgba(255,255,255,0.6); font-size: 0.85em; margin-top: 8px;">
        4 Specialized Agents: Case Search → Document Retrieval → Legal Reasoning → Summary
    </p>
</div>
"""

AGENT_PIPELINE_HTML = """
<div style="background: linear-gradient(135deg, #f7f5f0, #ede9de); border: 1px solid #d4c9b0;
     border-radius: 10px; padding: 16px 20px; margin-bottom: 18px; text-align: center;">
    <div style="font-weight: bold; color: #1a3a5c; margin-bottom: 10px; font-size: 1em; text-transform: uppercase; letter-spacing: 1px;">Agent Pipeline</div>
    <div style="display: flex; justify-content: center; align-items: center; flex-wrap: wrap; gap: 6px;">
        <span style="background: #1a3a5c; color: white; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: bold;">🔍 Case Search</span>
        <span style="color: #c9a84c; font-weight: bold; font-size: 1.2em;">→</span>
        <span style="background: #2e5983; color: white; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: bold;">📄 Document Retrieval</span>
        <span style="color: #c9a84c; font-weight: bold; font-size: 1.2em;">→</span>
        <span style="background: #1a5276; color: white; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: bold;">⚖️ Legal Reasoning</span>
        <span style="color: #c9a84c; font-weight: bold; font-size: 1.2em;">→</span>
        <span style="background: #c9a84c; color: #1a3a5c; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: bold;">📝 Summary Memo</span>
    </div>
</div>
"""

DISCLAIMER_HTML = """
<div style="background: #fff8e1; border: 1px solid #ffe082; border-left: 4px solid #f9a825;
     border-radius: 6px; padding: 10px 16px; margin-top: 8px; font-size: 0.85em; color: #5d4037;">
    ⚠️ <strong>Legal Disclaimer:</strong> This tool provides AI-assisted legal research for informational purposes only.
    It does not constitute legal advice. Always consult a licensed attorney for legal matters.
</div>
"""


# ============================================================
# GRADIO INTERFACE
# ============================================================
with gr.Blocks(
    css=CUSTOM_CSS,
    title="⚖️ AI Legal Research Agent",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.blue,
        secondary_hue=gr.themes.colors.amber,
        neutral_hue=gr.themes.colors.stone,
        font=[gr.themes.GoogleFont("Lora"), gr.themes.GoogleFont("Source Sans Pro"), "serif"],
    )
) as demo:

    # ── HEADER ──
    gr.HTML(HEADER_HTML)
    gr.HTML(AGENT_PIPELINE_HTML)

    with gr.Row():
        # ── LEFT PANEL: INPUT ──
        with gr.Column(scale=4):

            # Main Question Input
            gr.HTML("<div class='section-title'>📋 Legal Research Query</div>")
            legal_question_input = gr.Textbox(
                label="Legal Question",
                placeholder="Enter your legal question here...\n\nExample: What constitutes breach of fiduciary duty for corporate directors under Delaware law?",
                lines=4,
                max_lines=8,
                elem_id="legal-question"
            )

            gr.HTML("<div class='section-title' style='margin-top:16px;'>⚙️ Research Parameters</div>")

            with gr.Row():
                jurisdiction_dd = gr.Dropdown(
                    choices=[
                        "Federal", "U.S. Supreme Court", "California", "New York",
                        "Texas", "Delaware", "Florida", "Illinois", "Massachusetts",
                        "Washington D.C.", "International", "European Union"
                    ],
                    value="Federal",
                    label="🏛️ Jurisdiction",
                    info="Select the governing jurisdiction"
                )

                practice_area_dd = gr.Dropdown(
                    choices=[
                        "General Civil", "Contract Law", "Constitutional Law",
                        "Corporate Law", "Criminal Law", "Employment Law",
                        "Family Law", "Immigration Law", "Intellectual Property",
                        "Mergers & Acquisitions", "Real Estate Law", "Tax Law",
                        "Tort Law", "Securities Law", "Bankruptcy Law",
                        "Environmental Law", "Healthcare Law", "Antitrust Law"
                    ],
                    value="General Civil",
                    label="📚 Practice Area",
                    info="Select the area of law"
                )

            with gr.Row():
                research_depth_dd = gr.Dropdown(
                    choices=[
                        "Quick",        # 1 search
                        "Standard",     # 2 searches
                        "Deep",         # 3 searches
                        "Comprehensive" # 4 searches
                    ],
                    value="Standard",
                    label="🔬 Research Depth",
                    info="Deeper = more comprehensive but slower"
                )

                output_format_dd = gr.Dropdown(
                    choices=["Legal Memo (Markdown)", "Plain Text", "Detailed Report"],
                    value="Legal Memo (Markdown)",
                    label="📄 Output Format",
                    info="Format for the final memo"
                )

            gr.HTML("<div class='section-title' style='margin-top:16px;'>🔧 Research Options</div>")

            with gr.Row():
                include_news_cb = gr.Checkbox(
                    value=True,
                    label="📰 Include Recent Legal News",
                    info="Fetch recent court decisions and legal developments"
                )
                include_statutes_cb = gr.Checkbox(
                    value=True,
                    label="📜 Include Statutes & Regulations",
                    info="Look up applicable statutes and regulations"
                )

            # Action Buttons
            with gr.Row():
                research_btn = gr.Button(
                    "⚖️ Conduct Legal Research",
                    variant="primary",
                    size="lg",
                    elem_id="research-btn"
                )

            with gr.Row():
                clear_btn = gr.Button(
                    "🗑️ Clear All",
                    variant="secondary",
                    size="sm",
                    elem_id="clear-btn"
                )

            # Status Bar
            status_output = gr.Textbox(
                label="📊 Research Status",
                value="Ready — Enter a legal question to begin research.",
                interactive=False,
                lines=1
            )

            gr.HTML(DISCLAIMER_HTML)

        # ── RIGHT PANEL: EXAMPLES ──
        with gr.Column(scale=2):
            gr.HTML("<div class='section-title'>💼 Quick Examples</div>")
            gr.HTML("""
            <p style="font-size:0.85em; color:#666; margin-bottom:12px;">
            Click an example to auto-fill the research form:
            </p>
            """)

            example_buttons = []
            example_configs = [
                ("📝 Contract Breach (Federal)",   "Contract Breach (Federal)"),
                ("👔 Employment Discrimination",   "Employment Discrimination (CA)"),
                ("🏛️ 4th Amendment & Digital Data", "Fourth Amendment Search (SCOTUS)"),
                ("🏢 Corporate Director Liability",  "Corporate Liability (Delaware)"),
                ("💡 AI & Copyright Infringement",  "IP Copyright Infringement")
            ]

            for btn_label, example_key in example_configs:
                btn = gr.Button(
                    btn_label,
                    size="sm",
                    elem_id="example-btn"
                )
                example_buttons.append((btn, example_key))

            gr.HTML("""
            <div style="margin-top: 20px; background: #f0f4f8; border-radius: 8px; padding: 14px;
                 border: 1px solid #bee3f8; font-size: 0.83em;">
                <div style="font-weight: bold; color: #1565c0; margin-bottom: 8px;">🤖 AI Model Info</div>
                <div style="color: #444;">• Model: LLaMA 3.1 8B Instant</div>
                <div style="color: #444;">• Provider: Groq (Fast Inference)</div>
                <div style="color: #444;">• Search: Serper API</div>
                <div style="color: #444;">• Framework: LangGraph</div>
                <div style="color: #444; margin-top: 6px;">• 4 Specialized Agents</div>
                <div style="color: #444;">• 5 Legal Research Tools</div>
            </div>
            """)

            gr.HTML("""
            <div style="margin-top: 12px; background: #f0f9f0; border-radius: 8px; padding: 14px;
                 border: 1px solid #a5d6a7; font-size: 0.83em;">
                <div style="font-weight: bold; color: #2e7d32; margin-bottom: 8px;">⚡ Research Depth Guide</div>
                <div style="color: #444;">🟢 Quick: ~30s, 1 search</div>
                <div style="color: #444;">🟡 Standard: ~60s, 2 searches</div>
                <div style="color: #444;">🟠 Deep: ~90s, 3 searches</div>
                <div style="color: #444;">🔴 Comprehensive: ~2min+</div>
            </div>
            """)

    # ── OUTPUT SECTION ──
    gr.HTML("<hr style='border: 1px solid #d4c9b0; margin: 20px 0;'>")
    gr.HTML("<div class='section-title'>📊 Research Results</div>")

    with gr.Tabs():
        with gr.TabItem("📋 Legal Memo (Final Output)"):
            final_memo_output = gr.Markdown(
                value="*Research results will appear here after running...*",
                label="Legal Research Memorandum"
            )

        with gr.TabItem("🔍 Case Search Results"):
            search_output = gr.Markdown(
                value="*Case search results will appear here...*",
                label="Search Results (Agent 1)"
            )

        with gr.TabItem("📄 Retrieved Documents"):
            documents_output = gr.Markdown(
                value="*Retrieved legal documents will appear here...*",
                label="Document Repository (Agent 2)"
            )

        with gr.TabItem("⚖️ Legal Analysis (IRAC)"):
            analysis_output = gr.Markdown(
                value="*Legal reasoning analysis will appear here...*",
                label="IRAC Analysis (Agent 3)"
            )

    # ── FOOTER ──
    gr.HTML("""
    <div class="app-footer">
        ⚖️ AI Legal Research Agent | Powered by LangGraph + LLaMA 3.1 8B (Groq) + Serper API<br>
        Built for Law Firms • Compliance Teams • Legal Researchers
    </div>
    """)

    # ── WIRE UP EVENTS ──

    # Main research button
    research_btn.click(
        fn=process_legal_research,
        inputs=[
            legal_question_input,
            jurisdiction_dd,
            practice_area_dd,
            research_depth_dd,
            include_news_cb,
            include_statutes_cb,
            output_format_dd
        ],
        outputs=[
            final_memo_output,
            search_output,
            documents_output,
            analysis_output,
            status_output
        ],
        show_progress="full"
    )

    # Clear button
    clear_btn.click(
        fn=lambda: (
            "",
            "Federal", "General Civil", "Standard",
            "*Research results will appear here after running...*",
            "*Case search results will appear here...*",
            "*Retrieved legal documents will appear here...*",
            "*Legal reasoning analysis will appear here...*",
            "Ready — Enter a legal question to begin research."
        ),
        inputs=[],
        outputs=[
            legal_question_input,
            jurisdiction_dd, practice_area_dd, research_depth_dd,
            final_memo_output, search_output, documents_output,
            analysis_output, status_output
        ]
    )

    # Example buttons
    for btn, example_key in example_buttons:
        btn.click(
            fn=lambda key=example_key: load_example(key),
            inputs=[],
            outputs=[
                legal_question_input,
                jurisdiction_dd,
                practice_area_dd,
                research_depth_dd
            ]
        )


# ── LAUNCH ──
print("\n🚀 Launching AI Legal Research Agent UI...")
demo.launch(
    share=True,          # Creates a public Gradio link for Colab
    debug=False,
    show_error=True,
    quiet=False
)


🚀 Launching AI Legal Research Agent UI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20bd8478e548643f09.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
